# Energy DMRG (PEPO + PEPS)

Goal: minimize
`E(psi) = <psi|H|psi> / <psi|psi>`
with `pepsy.EnergyOptimizer`.

This version uses **clear, one-plot-per-cell diagnostics**, mostly on **log-scale** (similar style to `state_dmrg`).


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import quimb as qu
import quimb.tensor as qtn
from quimb.operator.pepobuilder import PEPO_nearest_neighbor

import pepsy as py
core = py.core
import torch

In [2]:
# Backend + contraction optimizer
to_backend = core.backend_torch(dtype=torch.complex128)
optimizer = core.build_optimizer(
    progbar=True,
    directory='cash/',
    parallel=False,
)

# Lattice / ITF model
Lx, Ly = 6, 6
J = 1.0
field = 3.0
dtype = 'complex128'
seed = 66
chi = 128
bond_dim = 2

# Sweep controls
n_sweeps = 5
env_n_iter = 10
n_round_trips = 1

# Local slice optimizer
solver = 'nlopt-lbfgs'
solver_options=dict(
    algorithm='LD_LBFGS',
    n_steps=20,
    maxeval=60,
    ftol_rel=1e-6,
    xtol_rel=1e-6,
    patience=20,
    min_steps=5,
    bad_max=5,
)
print({
    'Lx': Lx,
    'Ly': Ly,
    'chi': chi,
    'n_sweeps': n_sweeps,
    'env_n_iter': env_n_iter,
    'solver': solver,
})

{'Lx': 6, 'Ly': 6, 'chi': 128, 'n_sweeps': 5, 'env_n_iter': 10, 'solver': 'nlopt-lbfgs'}


## Build PEPO And Initial State

In [3]:
Z_mat = qu.pauli('Z')
X_mat = qu.pauli('X')

pepo = PEPO_nearest_neighbor(
    A=J * Z_mat,
    B=Z_mat,
    C=field * X_mat,
    Lx=Lx,
    Ly=Ly,
    cyclic=False,
    dtype=dtype,
)
pepo.add_tag('pepo')

state = qtn.PEPS.rand(Lx=Lx, Ly=Ly, bond_dim=3, seed=seed, dtype=dtype)

state.apply_to_arrays(to_backend)
pepo.apply_to_arrays(to_backend)

print('state shape:', (state.Lx, state.Ly), 'max bond:', state.max_bond())
print('pepo shape :', (pepo.Lx, pepo.Ly), 'max bond:', pepo.max_bond())
pepo.show()

state shape: (6, 6) max bond: 3
pepo shape : (6, 6) max bond: 3
  ╱ 3  ╱ 3  ╱ 3  ╱ 3  ╱ 3  ╱
 ●━━━━●━━━━●━━━━●━━━━●━━━━●
╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  
 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱
 ●━━━━●━━━━●━━━━●━━━━●━━━━●
╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  
 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱
 ●━━━━●━━━━●━━━━●━━━━●━━━━●
╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  
 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱
 ●━━━━●━━━━●━━━━●━━━━●━━━━●
╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  
 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱
 ●━━━━●━━━━●━━━━●━━━━●━━━━●
╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  ╱┃3  
 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱ 3 ┃╱
 ●━━━━●━━━━●━━━━●━━━━●━━━━●
╱    ╱    ╱    ╱    ╱    ╱    


In [ ]:
sweeper = py.EnergyOptimizer(
    state=state,
    pepo=pepo,
    chi=chi,
    contraction_opt=optimizer,
    fit_mode='eff',
)

print('E_boundary before:', sweeper.energy(n_iter=env_n_iter, progress=True, track_boundary_fidelity=False))

In [ ]:

sweep_cfg = dict(
    axes=('y', 'x'),
    n_round_trips=n_round_trips,
    chi=chi,
    optimizer=solver,
    optimizer_options=solver_options,
    env_n_iter=env_n_iter,
    progress=True,
    track_boundary_fidelity=False,
    renormalize=True,
)

sweeper.set_optimize_kwargs(**sweep_cfg)
result = sweeper.run(n_cycles=n_sweeps)

print('energy_before:', result['energy_before'])
print('energy_after :', result['energy_after'])
print('local updates:', len(result['runs']))
print('trace length :', len(result['energy'] or []))
print('fidels tracked:', len(sweeper.fidels))


In [ ]:
# MPO DMRG reference energy (same square ITF model)
H_mpo_sq, _H_pepo_sq = py.ham_tn.build_itf_lattice(
    L_x=Lx,
    L_y=Ly,
    lattice='square',
    cyclic=False,
    J=J,
    field=field,
)

dmrg = qtn.DMRG2(
    H_mpo_sq,
    bond_dims=[16, 32, 64],
    cutoffs=[1e-8, 1e-10, 1e-12],
)

converged = dmrg.solve(
    tol=1e-7,
    max_sweeps=6,
    verbosity=0,
)

E0 = float(np.real(dmrg.energy))
print('DMRG converged :', bool(converged))
print('DMRG sweeps    :', len(dmrg.energies))
print('DMRG E0 (real) :', E0)
print('GS MPS max bond:', dmrg.state.max_bond())

## Diagnostics (One Plot Per Cell, Log-Scale)

In [ ]:

runs = list(result['runs'])
steps = np.arange(1, len(runs) + 1, dtype=int)

move = np.array([str(r.get('sweep', '')) for r in runs])
is_fwd = move == 'forward'
is_bwd = move == 'backward'

energy_local = np.array([float(r.get('energy_final', np.nan)) for r in runs], dtype=float)
energy_ref_err = np.abs(energy_local - E0)
energy_final_err = np.abs(energy_local - float(result['energy_after']))

history_flat = []
move_end_idx = []
for r in runs:
    h = list(r.get('history') or [])
    if h:
        history_flat.extend(float(v) for v in h)
        move_end_idx.append(len(history_flat))
history_flat = np.array(history_flat, dtype=float)

fid_norm = np.array([
    np.nan if r.get('boundary_fidelity_norm') is None else float(r.get('boundary_fidelity_norm'))
    for r in runs
], dtype=float)
fid_energy = np.array([
    np.nan if r.get('boundary_fidelity_energy') is None else float(r.get('boundary_fidelity_energy'))
    for r in runs
], dtype=float)
bdy_inf_norm   = 1.0 - fid_norm
bdy_inf_energy = 1.0 - fid_energy

time_boundary = np.array([
    np.nan if r.get('time_boundary') is None else float(r.get('time_boundary'))
    for r in runs
], dtype=float)
time_optimize = np.array([
    np.nan if r.get('time_optimize') is None else float(r.get('time_optimize'))
    for r in runs
], dtype=float)
time_total = time_boundary + time_optimize
cum_time = np.nancumsum(np.nan_to_num(time_total, nan=0.0))

change_idx = np.where(move[1:] != move[:-1])[0] + 1

print('steps:', len(steps), '| fwd:', int(is_fwd.sum()), '| bwd:', int(is_bwd.sum()))
print('mean local energy :', float(np.nanmean(energy_local)))
print('mean t_bdy [s]    :', float(np.nanmean(time_boundary)))
print('mean t_opt [s]    :', float(np.nanmean(time_optimize)))


In [ ]:

import quimb as qu

# Plot 1: per-step local energy convergence.
with plt.style.context(qu.NEUTRAL_STYLE):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, energy_local, color='steelblue', lw=1.6, label='local energy per step')
    ax.axhline(result['energy_before'], color='gray',  ls='--', lw=1.0, label='E before')
    ax.axhline(result['energy_after'],  color='black', ls='-',  lw=1.0, label='E after')
    if 'E0' in dir():
        ax.axhline(E0, color='crimson', ls=':', lw=1.2, label='DMRG E0')
    for ci in change_idx:
        ax.axvline(ci - 0.5, color='0.80', lw=0.7)
    ax.set_title('Energy DMRG Convergence')
    ax.set_ylabel('Local energy')
    ax.set_xlabel('Step')
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    plt.show()

print('final local energy:', float(np.nanmean(energy_local[-4:])))


In [ ]:

# Plot 2: boundary fidelity per step (norm and energy).
fidels_norm    = [f['norm']   for f in sweeper.fidels]
fidels_energy  = [f['energy'] for f in sweeper.fidels]

with plt.style.context(qu.NEUTRAL_STYLE):
    fig, ax = plt.subplots(figsize=(9, 4))
    fsteps = range(1, len(sweeper.fidels) + 1)
    if any(v is not None for v in fidels_norm):
        ys = [v if v is not None else float('nan') for v in fidels_norm]
        ax.plot(fsteps, ys, color='orange', lw=1.6, label='boundary infidelity norm')
    if any(v is not None for v in fidels_energy):
        ys = [v if v is not None else float('nan') for v in fidels_energy]
        ax.plot(fsteps, ys, color='teal', lw=1.6, label='boundary infidelity energy')
    ax.set_title('Boundary Fidelity per Step')
    ax.set_ylabel('Boundary Infidelity')
    ax.set_xlabel('Step')
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    plt.show()


In [ ]:

# Plot 3: per-step timing (boundary MPS update vs gradient optimization).
with plt.style.context(qu.NEUTRAL_STYLE):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, time_boundary, color='sienna',    lw=1.6, label='boundary MPS time')
    ax.plot(steps, time_optimize, color='steelblue', lw=1.6, label='gradient opt time')
    for ci in change_idx:
        ax.axvline(ci - 0.5, color='0.80', lw=0.7)
    ax.set_title('Per-step Timing')
    ax.set_ylabel('Time (s)')
    ax.set_xlabel('Step')
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    plt.show()

print(f'mean t_bdy: {float(np.nanmean(time_boundary)):.3f}s  |  mean t_opt: {float(np.nanmean(time_optimize)):.3f}s')


## Solver switch examples

- SciPy L-BFGS-B:
  - `solver = "scipy"`
  - `solver_options = {"algorithm": "LBFGS", "n_steps": 50}`
- NLopt L-BFGS:
  - `solver = "nlopt"`
  - `solver_options = {"algorithm": "LD_LBFGS", "maxeval": 100}`
- Torch LBFGS:
  - `solver = "lbfgs"`
  - `solver_options = {"lr": 1e-1, "n_steps": 40}`
